In [1]:
!pip install pandas numpy faker snowflake-connector-python

In [2]:
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker('en_IN')
np.random.seed(42)

N_SELLERS    = 500
N_WAREHOUSES = 30
N_ORDERS     = 10000

cities    = ['Mumbai','Delhi','Bangalore','Hyderabad','Chennai',
             'Kolkata','Pune','Jaipur','Lucknow','Surat']
platforms = ['Amazon','Flipkart']
couriers  = ['BlueDart','Delhivery','Ekart','XpressBees','DTDC']

sellers = pd.DataFrame({
    'NODEID':       range(1, N_SELLERS+1),
    'SELLER_NAME':  [fake.company() for _ in range(N_SELLERS)],
    'PLATFORM':     np.random.choice(platforms, N_SELLERS),
    'CITY':         np.random.choice(cities, N_SELLERS),
    'GST_NUMBER':   [f"27{fake.bothify('??########Z#')}" for _ in range(N_SELLERS)],
    'BANK_ACCOUNT': [fake.bban() for _ in range(N_SELLERS)],
    'RETURN_RATE':  np.round(np.random.beta(2, 8, N_SELLERS), 4),
    'REG_DAYS_AGO': np.random.randint(30, 2000, N_SELLERS),
    'FRAUD_FLAG':   np.random.choice([0,1], N_SELLERS, p=[0.92, 0.08])
})

# Inject 10 fraud rings
for i in range(10):
    shared_bank = fake.bban()
    idx = np.random.choice(sellers.index, 5, replace=False)
    sellers.loc[idx, 'BANK_ACCOUNT'] = shared_bank
    sellers.loc[idx, 'FRAUD_FLAG']   = 1

warehouses = pd.DataFrame({
    'NODEID':         range(5001, 5001+N_WAREHOUSES),
    'WAREHOUSE_NAME': [f"WH-{c[:3].upper()}-{i}" for i,c
                       in enumerate(np.random.choice(cities, N_WAREHOUSES))],
    'CITY':           np.random.choice(cities, N_WAREHOUSES),
    'PIN_CODE':       [fake.postcode() for _ in range(N_WAREHOUSES)]
})

orders = pd.DataFrame({
    'SOURCENODEID':    np.random.choice(sellers['NODEID'], N_ORDERS),
    'TARGETNODEID':    np.random.choice(warehouses['NODEID'], N_ORDERS),
    'ORDER_ID':        [f"ORD{i:07d}" for i in range(N_ORDERS)],
    'ORDER_VALUE':     np.round(np.random.lognormal(8, 1.2, N_ORDERS), 2),
    'DELIVERY_DELAY':  np.random.choice([0,1,2,3,5,7,14], N_ORDERS,
                                         p=[.5,.2,.1,.08,.07,.03,.02]),
    'RETURN_CLAIMED':  np.random.choice([0,1], N_ORDERS, p=[0.87, 0.13]),
    'ROUTE_DEVIATION': np.random.choice([0,1], N_ORDERS, p=[0.95, 0.05])
})

logistics = pd.DataFrame({
    'SOURCENODEID': np.random.choice(warehouses['NODEID'], 200),
    'TARGETNODEID': range(9001, 9201),
    'COURIER':      np.random.choice(couriers, 200),
    'TRANSIT_DAYS': np.random.randint(1, 10, 200),
    'LOSS_COUNT':   np.random.poisson(0.3, 200)
})

print("✓ Sellers:",    sellers.shape)
print("✓ Warehouses:", warehouses.shape)
print("✓ Orders:",     orders.shape)
print("✓ Logistics:",  logistics.shape)

✓ Sellers: (500, 9)
✓ Warehouses: (30, 4)
✓ Orders: (10000, 7)
✓ Logistics: (200, 5)


In [3]:
import snowflake.connector

conn = snowflake.connector.connect(
    account   = 'UHZTCTQ-ERC65850',
    user      = 'PARASJAIN',
    password  = 'ParasJainSahab03',
    warehouse = 'SUPPLY_WH',
    database  = 'SUPPLY_CHAIN_DB',
    schema    = 'PUBLIC',
    role      = 'ACCOUNTADMIN'
)
cur = conn.cursor()
print("✓ Connected to Snowflake!")

✓ Connected to Snowflake!


In [4]:
from snowflake.connector.pandas_tools import write_pandas

tables = {
    'SELLERS':    sellers,
    'WAREHOUSES': warehouses,
    'ORDERS':     orders,
    'LOGISTICS':  logistics
}

for table_name, df in tables.items():
    success, nchunks, nrows, _ = write_pandas(
        conn, df, table_name,
        database  = 'SUPPLY_CHAIN_DB',
        schema    = 'PUBLIC',
        overwrite = False       # ← False this time, tables are fresh
    )
    print(f"✓ {table_name}: {nrows} rows | success={success}")

✓ SELLERS: 500 rows | success=True
✓ WAREHOUSES: 30 rows | success=True
✓ ORDERS: 10000 rows | success=True
✓ LOGISTICS: 200 rows | success=True


In [5]:
# Verify counts
for table in ['SELLERS','WAREHOUSES','ORDERS','LOGISTICS']:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"✓ {table}: {cur.fetchone()[0]} rows")

# Build shared bank account edge table
cur.execute("""
    CREATE OR REPLACE TABLE SHARED_BANK_EDGES AS
    SELECT
        s1.nodeId       AS sourceNodeId,
        s2.nodeId       AS targetNodeId,
        s1.bank_account AS shared_bank
    FROM SELLERS s1
    JOIN SELLERS s2
        ON  s1.bank_account = s2.bank_account
        AND s1.nodeId < s2.nodeId
""")
conn.commit()

cur.execute("SELECT COUNT(*) FROM SHARED_BANK_EDGES")
print(f"✓ Suspicious pairs: {cur.fetchone()[0]}")

✓ SELLERS: 1000 rows
✓ WAREHOUSES: 60 rows
✓ ORDERS: 20000 rows
✓ LOGISTICS: 400 rows
✓ Suspicious pairs: 184


In [6]:
cur.execute("""
    CREATE OR REPLACE TABLE SELLERS_GRAPH AS
    SELECT nodeId, return_rate, reg_days_ago, fraud_flag
    FROM SELLERS
""")

cur.execute("""
    CREATE OR REPLACE TABLE WAREHOUSES_GRAPH AS
    SELECT nodeId FROM WAREHOUSES
""")

cur.execute("""
    CREATE OR REPLACE TABLE ORDERS_GRAPH AS
    SELECT sourceNodeId, targetNodeId, order_value,
           delivery_delay, return_claimed, route_deviation
    FROM ORDERS
""")

cur.execute("""
    CREATE OR REPLACE TABLE SHARED_BANK_EDGES_GRAPH AS
    SELECT sourceNodeId, targetNodeId
    FROM SHARED_BANK_EDGES
""")

# Re-grant access to all tables including new ones
cur.execute("""
    GRANT SELECT, INSERT ON ALL TABLES IN SCHEMA SUPPLY_CHAIN_DB.PUBLIC
    TO DATABASE ROLE SUPPLY_CHAIN_DB.gds_db_role
""")

conn.commit()
print("✓ Graph tables created!")
print("✓ Permissions granted!")

✓ Graph tables created!
✓ Permissions granted!


In [7]:
cur.execute("SHOW TABLES IN SUPPLY_CHAIN_DB.PUBLIC")
tables = cur.fetchall()
print(f"✓ Total tables: {len(tables)}")
for t in tables:
    print(f"   → {t[1]}")


✓ Total tables: 10
   → LOGISTICS
   → ORDERS
   → ORDERS_GRAPH
   → SELLERS
   → SELLERS_GRAPH
   → SELLER_COMMUNITIES
   → SHARED_BANK_EDGES
   → SHARED_BANK_EDGES_GRAPH
   → WAREHOUSES
   → WAREHOUSES_GRAPH


In [8]:
for table in ['SELLERS', 'WAREHOUSES', 'ORDERS', 'LOGISTICS']:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"✓ {table}: {count} rows in Snowflake")

✓ SELLERS: 1000 rows in Snowflake
✓ WAREHOUSES: 60 rows in Snowflake
✓ ORDERS: 20000 rows in Snowflake
✓ LOGISTICS: 400 rows in Snowflake


In [9]:
cur.execute("""
    CREATE OR REPLACE TABLE SHARED_BANK_EDGES AS
    SELECT
        s1.nodeId       AS sourceNodeId,
        s2.nodeId       AS targetNodeId,
        s1.bank_account AS shared_bank
    FROM SELLERS s1
    JOIN SELLERS s2
        ON  s1.bank_account = s2.bank_account
        AND s1.nodeId < s2.nodeId
""")
conn.commit()
cur.execute("SELECT COUNT(*) FROM SHARED_BANK_EDGES")
print(f"✓ Suspicious pairs: {cur.fetchone()[0]}")


✓ Suspicious pairs: 184


In [10]:
for table in ['SELLERS', 'WAREHOUSES', 'ORDERS', 'LOGISTICS', 'SHARED_BANK_EDGES']:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"✓ {table}: {count} rows")


✓ SELLERS: 1000 rows
✓ WAREHOUSES: 60 rows
✓ ORDERS: 20000 rows
✓ LOGISTICS: 400 rows
✓ SHARED_BANK_EDGES: 184 rows
